# Tunda Box – Junior Data Analyst Assessment: Solutions
**Candidate:** Erick Odiwuor  |  **Date:** 5 November 2026

All data is fictional. This notebook covers the SQL/Python work (Part 2), the data-quality audit and spreadsheet logic (Part 3), the statistics (Part 4) and the business case (Part 5). The written answers for Parts 1 and 6 are in the response document linked in the README.

**Approach.** I load the two sample tables into an in-memory SQLite database so the SQL runs as written. Throughout I state my assumptions, because the sample has several deliberate data-quality traps (see Part 3, D1). Where a trap changes an answer, I show both the naive and the cleaned result.

In [1]:
import sqlite3, math
import pandas as pd
from scipy.stats import norm

customers = pd.DataFrame({
 "customer_id":[101,102,103,104,105,106,107,108,109,110],
 "city":["Nairobi","nairobi","Mombasa","Kisumu","Nairobi ","Nakuru","Mombasa","Nairobi","Eldoret","Kisumu"],
 "signup_date":["2026-01-14","2026-02-03","2026-06-20","2026-03-02","2026-03-15","2026-04-01","2026-04-18","2026-05-09","2026-06-21","2026-07-05"],
 "channel":["Organic","Referral","Paid Social","Organic","Paid Social","Referral","Organic","Paid Social","Organic","Paid Social"]})

rows = [
(1001,101,"2026-06-03","completed",2400,"M-Pesa"),(1002,102,"2026-06-05","completed",1850,"M-Pesa"),
(1003,103,"2026-06-11","cancelled",3200,"Card"),(1004,104,"2026-06-18","completed",2100,"M-Pesa"),
(1005,101,"2026-06-25","completed",2650,"Card"),(1006,105,"2026-07-02","completed",1500,"M-Pesa"),
(1006,105,"2026-07-02","completed",1500,"M-Pesa"),(1007,106,"2026-07-09","refunded",2900,"Card"),
(1008,102,"2026-07-14","completed",3100,"M-Pesa"),(1009,107,"2026-07-21","completed",None,"M-Pesa"),
(1010,101,"2026-07-28","completed",2200,"M-Pesa"),(1011,108,"2026-08-04","completed",1750,"Card"),
(1012,103,"2026-08-09","completed",2800,"M-Pesa"),(1013,104,"2026-08-15","completed",185000,"M-Pesa"),
(1014,105,"2026-08-19","cancelled",1600,"M-Pesa"),(1015,102,"2026-08-27","completed",2450,"M-Pesa"),
(1016,106,"2026-09-03","completed",3300,"Card"),(1017,101,"2026-09-08","completed",2750,"M-Pesa"),
(1018,108,"2026-09-14","refunded",2000,"M-Pesa"),(1019,107,"2026-09-22","completed",2350,"Card"),
(1020,999,"2026-09-26","completed",1900,"M-Pesa")]
orders = pd.DataFrame(rows, columns=["order_id","customer_id","order_date","status","amount_kes","payment_method"])

con = sqlite3.connect(":memory:")
customers.to_sql("customers", con, index=False)
orders.to_sql("orders", con, index=False)
def q(sql): return pd.read_sql(sql, con)
print(len(orders), "order rows,", orders.order_id.nunique(), "distinct order_ids")

21 order rows, 20 distinct order_ids


# Part 2: SQL and Python

## S1. Orders by status
**Assumption:** order 1006 appears twice with identical values, so it is a duplicate row, not two orders. I count distinct orders and show the raw count too.

In [2]:
display(q("SELECT status, COUNT(*) AS orders FROM orders GROUP BY status ORDER BY orders DESC"))   # raw, includes duplicate row
display(q("""
SELECT status, COUNT(*) AS orders
FROM (SELECT DISTINCT * FROM orders)
GROUP BY status ORDER BY orders DESC"""))

,status,orders
0,completed,17
1,refunded,2
2,cancelled,2


,status,orders
0,completed,16
1,refunded,2
2,cancelled,2


**Result (after removing the duplicate 1006 row):** completed 16, refunded 2, cancelled 2 = **20 orders**. The raw count is 21 rows (completed 17), so the duplicate would overstate completed orders by one.

## S2. Revenue by city
**(a)** "Completed-order revenue" = `status = 'completed'`. Cleaning applied: dedupe 1006, standardise city with `LOWER(TRIM(city))` (so "Nairobi", "nairobi" and "Nairobi " are one city), join to customers (this drops orphan order 1020, whose customer 999 does not exist). Order 1013 (KES 185,000) is a suspected typo, so I show the answer with and without it.

In [3]:
base = """
SELECT LOWER(TRIM(c.city)) AS city, SUM(o.amount_kes) AS completed_revenue_kes, COUNT(*) AS orders
FROM (SELECT DISTINCT * FROM orders) o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.status = 'completed' {extra}
GROUP BY 1 ORDER BY 2 DESC"""
print("Naive (no city cleaning, no dedupe):")
display(q("SELECT c.city, SUM(o.amount_kes) AS rev FROM orders o JOIN customers c USING(customer_id) WHERE o.status='completed' GROUP BY c.city ORDER BY rev DESC"))
print("Cleaned, order 1013 kept as recorded:");  display(q(base.format(extra="")))
print("Cleaned, order 1013 excluded as a suspected typo:"); display(q(base.format(extra="AND o.order_id <> 1013")))

Naive (no city cleaning, no dedupe):


,city,rev
0,Kisumu,187100.0
1,Nairobi,11750.0
2,nairobi,7400.0
3,Mombasa,5150.0
4,Nakuru,3300.0
5,Nairobi,3000.0


Cleaned, order 1013 kept as recorded:


,city,completed_revenue_kes,orders
0,kisumu,187100.0,2
1,nairobi,20650.0,9
2,mombasa,5150.0,3
3,nakuru,3300.0,1


Cleaned, order 1013 excluded as a suspected typo:


,city,completed_revenue_kes,orders
0,nairobi,20650.0,9
1,mombasa,5150.0,3
2,nakuru,3300.0,1
3,kisumu,2100.0,1


**Result:** **Nairobi, KES 20,650** (9 completed orders) once city names are standardised and the duplicate is removed. Without cleaning, the Nairobi revenue is split across three labels ("Nairobi", "nairobi", "Nairobi ") and the 185,000 order makes Kisumu look like the winner (KES 187,100).

**(b) What I send my manager in ten minutes:**
> "Nairobi leads completed-order revenue at KES 20,650 (next: Mombasa 5,150). I standardised city spellings and removed one duplicate order. One caveat: order 1013 (Kisumu, KES 185,000) looks like a typo for about 1,850. If it is real, Kisumu would lead at ~187k, so I'm confirming it with whoever owns the orders data and will update you. Also excluded: one order with no matching customer (KES 1,900) and one completed order with a missing amount."

The point: send the number now, with the caveat and a named next step, rather than wait for perfect data or send a wrong number silently.

## S3. Customers who never ordered
Anti-join (`LEFT JOIN ... WHERE o.order_id IS NULL`). I avoid `NOT IN` because it returns nothing if the subquery contains a NULL.

In [4]:
display(q("""
SELECT c.customer_id, c.city, c.channel, c.signup_date
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL"""))

,customer_id,city,channel,signup_date
0,109,Eldoret,Organic,2026-06-21
1,110,Kisumu,Paid Social,2026-07-05


**Result:** customers **109** (Eldoret, Organic) and **110** (Kisumu, Paid Social). Both signed up recently (June and July 2026), so they may simply not have had time to order yet.

## S4. Repeat purchase rate
**Definition (my assumption):** a repeat buyer is a customer with **2 or more distinct completed orders**. The denominator is customers with **at least one completed order**, since cancelled and refunded orders are not real purchases. I exclude unknown customer 999. The question could also mean "% of all registered customers", so I give that too.

In [5]:
display(q("""
WITH d AS (SELECT DISTINCT * FROM orders),
per_customer AS (
  SELECT o.customer_id, COUNT(*) AS completed_orders
  FROM d o JOIN customers c ON c.customer_id = o.customer_id
  WHERE o.status = 'completed'
  GROUP BY o.customer_id)
SELECT COUNT(*) AS buyers,
       SUM(completed_orders >= 2) AS repeat_buyers,
       ROUND(100.0 * SUM(completed_orders >= 2) / COUNT(*), 1) AS repeat_rate_pct,
       ROUND(100.0 * SUM(completed_orders >= 2) / (SELECT COUNT(*) FROM customers), 1) AS pct_of_all_customers
FROM per_customer"""))

,buyers,repeat_buyers,repeat_rate_pct,pct_of_all_customers
0,8,4,50.0,40.0


**Result: 50% (4 of 8 buyers).** Repeat buyers are 101 (4 orders), 102 (3), 104 (2) and 107 (2). As a share of all 10 registered customers it is 40%.

**Working:** (1) dedupe orders, (2) keep completed orders for known customers, (3) count orders per customer, (4) divide customers with 2+ by customers with 1+. Skipping the dedupe would wrongly turn customer 105 into a repeat buyer (duplicate 1006) and give 62.5%. Counting cancelled and refunded orders would give 88.9%. Both are overstated. With only 8 buyers in a sample extract, treat 50% as indicative, not a stable KPI.

## S5. Month-on-month revenue (completed)
Approach: dedupe, filter completed, aggregate by `substr(order_date,1,7)`, then use the `LAG()` window function for the previous month and compute `(rev - prev) / prev`. Months are only June to September because that is what the extract holds. `SUM` ignores the NULL amount on order 1009.

In [6]:
mom = """
WITH d AS (SELECT DISTINCT * FROM orders {where}),
m AS (
  SELECT substr(order_date,1,7) AS month, SUM(amount_kes) AS revenue_kes
  FROM d WHERE status = 'completed' GROUP BY 1)
SELECT month, revenue_kes,
       LAG(revenue_kes) OVER (ORDER BY month) AS prev_month,
       ROUND(100.0 * (revenue_kes - LAG(revenue_kes) OVER (ORDER BY month))
             / LAG(revenue_kes) OVER (ORDER BY month), 1) AS pct_change
FROM m"""
print("As recorded (includes the 185,000 order):"); display(q(mom.format(where="")))
print("With suspect order 1013 excluded:"); display(q(mom.format(where="WHERE order_id <> 1013")))

As recorded (includes the 185,000 order):


,month,revenue_kes,prev_month,pct_change
0,2026-06,9000.0,NaN,NaN
1,2026-07,6800.0,9000.0,-24.4
2,2026-08,192000.0,6800.0,2723.5
3,2026-09,10300.0,192000.0,-94.6


With suspect order 1013 excluded:


,month,revenue_kes,prev_month,pct_change
0,2026-06,9000.0,NaN,NaN
1,2026-07,6800.0,9000.0,-24.4
2,2026-08,7000.0,6800.0,2.9
3,2026-09,10300.0,7000.0,47.1


**Result (recommended view, order 1013 excluded):** Jun 9,000 → Jul 6,800 (**−24.4%**) → Aug 7,000 (**+2.9%**) → Sep 10,300 (**+47.1%**).
As recorded, August is 192,000 (**+2,723.5%**) and September then falls **−94.6%**, which is an artefact of one probable typo, not a business trend. If 1013 is really 1,850, August is 8,850 (+30.1%) and September +16.4%. Either way September is the strongest month. September includes order 1020 (KES 1,900) even though its customer is unknown, because the money is real.

# Part 3: Data quality and spreadsheets

## D1. Audit the data
Checks run below, then the issues and what I'd do about each.

In [7]:
d = orders.copy()
print("Duplicate rows:\n", d[d.duplicated(keep=False)])
print("\nNULL amounts:\n", d[d.amount_kes.isna()])
print("\nAmount outliers (> 10x median):\n", d[d.amount_kes > 10*d.amount_kes.median()])
print("\nOrphan customer_ids:\n", d[~d.customer_id.isin(customers.customer_id)])
m = d.merge(customers, on="customer_id")
print("\nOrders dated before signup:\n", m[m.order_date < m.signup_date][["order_id","customer_id","order_date","signup_date","status"]])
print("\nCity labels with whitespace/case issues:\n", customers[customers.city != customers.city.str.strip().str.title()])

Duplicate rows:
    order_id  customer_id  order_date     status  amount_kes payment_method
5      1006          105  2026-07-02  completed      1500.0         M-Pesa
6      1006          105  2026-07-02  completed      1500.0         M-Pesa

NULL amounts:
    order_id  customer_id  order_date     status  amount_kes payment_method
9      1009          107  2026-07-21  completed         NaN         M-Pesa

Amount outliers (> 10x median):
     order_id  customer_id  order_date     status  amount_kes payment_method
13      1013          104  2026-08-15  completed    185000.0         M-Pesa

Orphan customer_ids:
     order_id  customer_id  order_date     status  amount_kes payment_method
20      1020          999  2026-09-26  completed      1900.0         M-Pesa

Orders dated before signup:
    order_id  customer_id  order_date signup_date     status
2      1003          103  2026-06-11  2026-06-20  cancelled

City labels with whitespace/case issues:
    customer_id      city signup_date  

| # | Issue | Where | What I'd do |
|---|---|---|---|
| 1 | **Duplicate order row** | Order 1006 appears twice, identical | Dedupe on `order_id`; ask engineering why the pipeline double-loads; add a uniqueness check. |
| 2 | **Inconsistent city values** | "nairobi" (102), "Nairobi " with trailing space (105) | Standardise with `TRIM`/`LOWER` (or `PROPER`) in the cleaning step; better, fix at source with a controlled list of cities. |
| 3 | **Extreme outlier** | Order 1013, KES 185,000 (next largest is 3,300) | Do not silently delete. Verify with the source system. Likely a typo for ~1,850 (extra zeros), so correct if confirmed, else flag and report with and without it. |
| 4 | **Missing amount (NULL)** | Order 1009, status completed | Find the true amount from payments/M-Pesa records; until then report revenue as "excluding 1 order with missing amount", never treat it as zero. |
| 5 | **Orphan order** | Order 1020 points to customer 999, who is not in `customers` | Inner joins silently drop it. Check whether customers is incomplete or the ID is wrong; keep it in total revenue but flag it in customer-level analysis. |
| 6 | **Order before signup** | Order 1003 (11 Jun) precedes customer 103's signup (20 Jun) | Check whether `signup_date` is overwritten on re-registration, or guest checkout is merged later. It affects cohort and retention analysis. |
| 7 | **Status affects every metric** | Cancelled (2) and refunded (2) orders have positive amounts | Agree one written definition of "revenue" (completed only? net of refunds?) before reporting. Refunds should probably be netted off. |
| 8 | **It is a sample** | Only 10 customers, 20 orders, Jun–Sep 2026 | Do not generalise. Re-run on the full extract and quantify how widespread each issue is before sharing. |

**Before sharing:** fix or flag 1–5, document definitions, and put the checks into a repeatable script so the audit is not a one-off.

## D2. Spreadsheet formulas
Orders sheet columns: A order_id, B customer_id, C order_date, D status, E amount_kes, F payment_method. Customers sheet: A customer_id, B city.

**Total completed M-Pesa revenue**
```
=SUMIFS(E:E, D:D, "completed", F:F, "M-Pesa")
```
`SUMIFS` ignores text such as "NULL" and blanks. It will still include the duplicate 1006 row (KES 209,550 as listed; 208,050 after dedupe), and the 185,000 outlier, so clean first.

**City for each order** (in G2, filled down)
```
=IFERROR(TRIM(XLOOKUP(B2, customers!A:A, customers!B:B)), "Unknown customer")
```
Older Excel / Google Sheets equivalent:
```
=IFERROR(TRIM(VLOOKUP(B2, customers!A:B, 2, FALSE)), "Unknown customer")
```
`FALSE` forces an exact match. `TRIM` removes the stray space in "Nairobi ", and `IFERROR` handles the orphan customer 999 instead of showing `#N/A`.

In [8]:
cleaned = orders.drop_duplicates()
print("M-Pesa completed, as listed:", orders[(orders.status=="completed")&(orders.payment_method=="M-Pesa")].amount_kes.sum())
print("M-Pesa completed, deduplicated:", cleaned[(cleaned.status=="completed")&(cleaned.payment_method=="M-Pesa")].amount_kes.sum())

M-Pesa completed, as listed: 209550.0
M-Pesa completed, deduplicated: 208050.0


## D3. Mean or median

In [9]:
c = orders.drop_duplicates().query("status == 'completed'").amount_kes.dropna()
print(f"n={len(c)}  mean={c.mean():,.0f}  median={c.median():,.0f}")
c2 = c[c != 185000]
print(f"Without order 1013: mean={c2.mean():,.0f}  median={c2.median():,.0f}")

n=15  mean=14,540  median=2,400
Without order 1013: mean=2,364  median=2,375


**Median.** Order values are right-skewed and one record (185,000) drags the mean to about **KES 14,540**, which no actual customer pays. The median is about **KES 2,400** and sits right where the other orders are. Without the outlier, mean (2,364) and median (2,375) agree, which shows the median was the more robust choice all along. I'd report the median as the "typical" order value, optionally alongside the mean and a note on outliers, once the 185,000 record is verified.

# Part 4: Statistics

## T1. Concepts in plain language
**(a) p-value for a manager.** "Suppose the change we made actually did nothing. The p-value is how often we'd still see a result at least as big as this one just by luck. A small p-value (conventionally under 0.05) means 'that would be very unusual if nothing were going on', so we have evidence the change is real. It does **not** tell us how big or how important the effect is, nor the probability that we're right."

**(b) Do SMS offers cause the extra spending?** **Not necessarily.** That is a correlation from observational data. Possible confounders: the business may *choose* to text its best or most active customers (selection bias), or customers who already like us opt in to SMS (self-selection), so they would have spent more anyway. Reverse causality is also possible. To establish causation I'd run a **randomised test**: randomly hold out a share of eligible customers from SMS and compare spend. If a test isn't possible, I'd compare similar customers (matching or regression controlling for past spend and tenure) and be clear that this reduces, but does not remove, doubt.

## T2. Read an A/B test

In [10]:
n1=n2=1000; x1,x2=50,60
p1,p2=x1/n1,x2/n2
pool=(x1+x2)/(n1+n2)
z=(p2-p1)/math.sqrt(pool*(1-pool)*(1/n1+1/n2))
p_value=2*(1-norm.cdf(abs(z)))
se=math.sqrt(p1*(1-p1)/n1+p2*(1-p2)/n2)
lo,hi=(p2-p1)-1.96*se,(p2-p1)+1.96*se
print(f"Conversion: {p1:.1%} vs {p2:.1%}  | absolute lift {p2-p1:.1%} | relative lift {(p2-p1)/p1:.0%}")
print(f"z = {z:.2f}, two-sided p-value = {p_value:.2f}")
print(f"95% CI for difference: [{lo:+.1%}, {hi:+.1%}]")
n_needed = 2*((1.96+0.8416)**2)*((p1*(1-p1)+p2*(1-p2))/2)/(0.01**2)
print(f"Users per arm to detect a 1-point lift with 80% power: ~{n_needed:,.0f}")

Conversion: 5.0% vs 6.0%  | absolute lift 1.0% | relative lift 20%
z = 0.98, two-sided p-value = 0.33
95% CI for difference: [-1.0%, +3.0%]
Users per arm to detect a 1-point lift with 80% power: ~8,155


**Recommendation: do not ship on this evidence yet.** The "20% rise" is the *relative* change (5.0% → 6.0%), but it is only 10 extra conversions out of 1,000 users. A two-proportion z-test gives z ≈ 0.98 and **p ≈ 0.33**, far above 0.05, so a gap this size appears about a third of the time by chance alone. The 95% confidence interval for the true lift runs from about **−1.0 to +3.0 percentage points**, so the new checkout could even be worse. I'd keep the test running until each arm has roughly **8,000+ users** (enough to detect a 1-point lift with 80% power), check there's no sample-ratio mismatch or novelty effect, and look at revenue per visitor as well as conversion. If shipping is cheap and low-risk and the team accepts the uncertainty, that is a business call, but it should not be presented as a proven win.

# Part 5: Business case – falling conversion

In [11]:
t = pd.DataFrame({"channel":["Organic","Paid Social"],
                  "sep_sessions":[40000,10000],"sep_orders":[2400,200],
                  "oct_sessions":[38000,30000],"oct_orders":[2318,630]})
t["sep_cr"]=t.sep_orders/t.sep_sessions; t["oct_cr"]=t.oct_orders/t.oct_sessions
t["sep_share"]=t.sep_sessions/t.sep_sessions.sum(); t["oct_share"]=t.oct_sessions/t.oct_sessions.sum()
display(t.style.format({c:"{:.1%}" for c in ["sep_cr","oct_cr","sep_share","oct_share"]}))
tot=lambda m: t[f"{m}_orders"].sum()/t[f"{m}_sessions"].sum()
print(f"Blended conversion: Sep {tot('sep'):.2%} -> Oct {tot('oct'):.2%}")
print(f"Total orders: {t.sep_orders.sum():,} -> {t.oct_orders.sum():,} ({t.oct_orders.sum()/t.sep_orders.sum()-1:+.1%})")
print(f"Oct blended rate if channel mix were still 80/20: {0.8*t.oct_cr[0]+0.2*t.oct_cr[1]:.2%}")

,channel,sep_sessions,sep_orders,oct_sessions,oct_orders,sep_cr,oct_cr,sep_share,oct_share
0,Organic,40000,2400,38000,2318,6.0%,6.1%,80.0%,55.9%
1,Paid Social,10000,200,30000,630,2.0%,2.1%,20.0%,44.1%


Blended conversion: Sep 5.20% -> Oct 4.34%
Total orders: 2,600 -> 2,948 (+13.4%)
Oct blended rate if channel mix were still 80/20: 5.30%


### (a) What is happening, and what else I'd want to know
**This is a mix-shift effect (Simpson's-paradox pattern), not paid social breaking the site.**
- Blended conversion did fall (5.2% → 4.3%), **but conversion improved in both channels**: Organic 6.0% → 6.1% and Paid Social 2.0% → 2.1%.
- What changed is the traffic mix. Paid Social went from 20% to **44%** of sessions (10k → 30k). Paid traffic converts at roughly a third of the organic rate, so more of it pulls the blended average down. If the mix had stayed 80/20, October would have been about **5.3%**.
- Total orders **rose 13.4%** (2,600 → 2,948). Paid Social added **430 extra orders** (200 → 630). Pausing it would remove about 630 orders, taking October to roughly 2,318.
- Not explained by paid social: **Organic sessions fell 5%** (40k → 38k), worth investigating (seasonality, SEO ranking, brand searches, cannibalisation by paid).

**What else I'd want to know:** paid social spend and **cost per order (CAC)** vs margin; average order value and gross margin by channel; **repeat rate and lifetime value** of paid-acquired customers; whether the October campaign targeted new audiences or placements; bounce rate / landing-page and device mix of paid traffic; whether tracking or attribution changed between months; refund and cancellation rates by channel; and whether Organic's drop is linked to paid (e.g. branded search shifting).

### (b) Recommendation (two sentences)
> "Please don't pause paid social: conversion rose in both channels, and the headline drop is because paid traffic (which naturally converts lower) grew from 20% to 44% of visits, while it added about 430 orders. Instead, judge paid social on cost per order and customer lifetime value, and let me check why organic sessions slipped 5%."

---
## Part 6 and Part 1
Written answers (experience, conflicting numbers, questions for Tunda Box) are in the response document referenced in the README.